In [ ]:
# Job configuration and helpers
import codecs
import os
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path

BRANCH = 'feature/sota-baselines-bcic2a'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
TARGET_SUBJECTS = list(range(1, 10))
JOB_TAG = 's01_s09'

REPO_PATH = Path(f"/kaggle/working/eegconformer-bcic2a-{JOB_TAG}")
MNE_DATA = Path("/kaggle/working/mne_data")
TRAIN_LOG = Path(f"/kaggle/working/eegconformer_bcic2a_{JOB_TAG}.log")
RESULT_ARCHIVE = Path(f"/kaggle/working/eegconformer_bcic2a_{JOB_TAG}_results")


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print("+", " ".join(command), flush=True)
    if not stream:
        subprocess.run(
            command,
            cwd=str(cwd) if cwd else None,
            env=env,
            check=True,
        )
        return

    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    started_at = time.monotonic()
    with open(log_path, "w", encoding="utf-8") as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end="", flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f"[Kaggle heartbeat] running | elapsed={elapsed:.1f}m\n"
                print(heartbeat, end="", flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break

        remaining = decoder.decode(b"", final=True)
        if remaining:
            print(remaining, end="", flush=True)
            log_file.write(remaining)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Training exited with code {return_code}. Full log: {log_path}"
        )

# Clone the exact source-only best-validation branch and install its pinned environment
run([sys.executable, "-m", "pip", "install", "-q", "uv"])

if (REPO_PATH / ".git").is_dir():
    run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_PATH)
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_PATH)
    run(["git", "checkout", "--force", BRANCH], cwd=REPO_PATH)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_PATH)
else:
    run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPO_URL, REPO_PATH,
    ])

run(["git", "log", "-1", "--oneline"], cwd=REPO_PATH)

# Fail early if this is not actually the best-validation implementation.
pipeline_text = (REPO_PATH / "train_pipeline.py").read_text(encoding="utf-8")
assert "ModelCheckpoint" in pipeline_text
assert 'monitor="val_acc"' in pipeline_text
assert "best_model_path" in pipeline_text
config_text = (REPO_PATH / "configs/eegconformer_loso.yaml").read_text(encoding="utf-8")
assert "domain_adaptation: false" in config_text
assert "riemannian_alignment: false" in config_text
assert 'model: "EEGConformer"' in config_text

model_text = (REPO_PATH / "models/eegconformer.py").read_text(encoding="utf-8")
assert "class EEGConformerModule(nn.Module)" in model_text
print("Verified: EEG Conformer reference architecture, pure source-only protocol.")
print("Verified: pure source-only graph and best source-validation checkpoint selection.")

UV = shutil.which("uv") or "uv"
run([UV, "venv", "--clear", "--python", "3.10", ".venv"], cwd=REPO_PATH)
PYTHON = REPO_PATH / ".venv/bin/python"
run([
    UV, "pip", "install", "--python", PYTHON,
    "torch==2.7.1", "torchvision==0.22.1",
    "--index-url", "https://download.pytorch.org/whl/cu126",
])
run([UV, "pip", "install", "--python", PYTHON, "-r", "requirements.txt"], cwd=REPO_PATH)

# Preserve model/data defaults; restrict targets and compact epoch logging.
override = (
    "import yaml; from pathlib import Path; "
    "p=Path('configs/eegconformer_loso.yaml'); "
    "c=yaml.safe_load(p.read_text()); "
    f"c['subject_ids']={TARGET_SUBJECTS!r}; "
    "c['log_every_n_epochs']=10; "
    "c['enable_progress_bar']=False; "
    "p.write_text(yaml.safe_dump(c, sort_keys=False))"
)
run([PYTHON, "-c", override], cwd=REPO_PATH)

run(["nvidia-smi"])
run([
    PYTHON, "-c",
    "import torch; print('PyTorch:', torch.__version__); "
    "print('CUDA:', torch.cuda.is_available()); "
    "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'; "
    "print('GPU:', torch.cuda.get_device_name(0))",
])

# Train only this job's LOSO target subjects and export its results
MNE_DATA.mkdir(parents=True, exist_ok=True)

# A rerun in the same draft session must not mix old and new result folders.
results_dir = REPO_PATH / "results"
if results_dir.exists():
    shutil.rmtree(results_dir)

environment = os.environ.copy()
environment.update({
    "PYTHONUNBUFFERED": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "MPLBACKEND": "Agg",
    "MNE_DATA": str(MNE_DATA),
    "MNE_DATASETS_BNCI_PATH": str(MNE_DATA),
})

print(
    "===== BCI IV-2a EEG CONFORMER SOURCE-ONLY BEST-VAL LOSO | "
    f"TARGETS={TARGET_SUBJECTS} =====",
    flush=True,
)
print("Live log:", TRAIN_LOG, flush=True)

run([
    PYTHON, "-u", "train_pipeline.py",
    "--model", "eegconformer_loso",
    "--dataset", "bcic2a",
    "--loso",
    "--gpu_id", "0",
], cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG)

archive = shutil.make_archive(
    str(RESULT_ARCHIVE),
    "zip",
    root_dir=REPO_PATH,
    base_dir="results",
)
print("Completed targets:", TARGET_SUBJECTS)
print("Kaggle output archive:", archive)
print("Kaggle output log:", TRAIN_LOG)
